# CỰC KỲ QUAN TRỌNG !!!
# gkinhere cần học lại kỹ các code, logic, chiến lược crawl ở file này nhé 

In [ ]:
# Cell 1: Imports
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from bs4 import BeautifulSoup
import time
import json
from datetime import datetime, timedelta
import re
import pandas as pd
import numpy as np
import random
from urllib.parse import urljoin

import undetected_chromedriver as uc

# For PostgreSQL
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

In [ ]:
# Cell 2: Constants and Configuration
TARGET_URL_BASE = "https://www.topcv.vn"
INITIAL_TARGET_URL = "https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?type_keyword=0&sba=1&locations=l2"
MAX_PAGES_TO_SCRAPE = 15

TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
JSON_OUTPUT_FILENAME = f"topcv_all_jobs_data_{TIMESTAMP_STR}.json"

# Danh sách các quận/huyện của TP.HCM để chuẩn hóa và khớp
HCM_DISTRICTS_FULL_LIST = [
    "Quận 1", "Quận 2", "Quận 3", "Quận 4", "Quận 5", "Quận 6", "Quận 7", "Quận 8",
    "Quận 9", "Quận 10", "Quận 11", "Quận 12",
    "Thủ Đức", "Bình Thạnh", "Tân Bình", "Tân Phú", "Phú Nhuận", "Gò Vấp", "Bình Tân",
    "Hóc Môn", "Củ Chi", "Nhà Bè", "Bình Chánh", "Cần Giờ",
    "Thành phố Thủ Đức" # Bao gồm cả "Thành phố Thủ Đức"
]
# Chuẩn hóa danh sách quận để khớp không phân biệt chữ hoa/thường và các tiền tố
HCM_DISTRICTS_NORMALIZED_FOR_MATCH = [
    d.lower().replace("quận ", "").replace("huyện ", "").replace("thành phố ", "").replace("tp. ", "")
    for d in HCM_DISTRICTS_FULL_LIST
]


print(f"URL Mục tiêu ban đầu: {INITIAL_TARGET_URL}")
print(f"Số trang tối đa sẽ cào (theo cấu hình): {MAX_PAGES_TO_SCRAPE}")
print(f"File JSON tổng hợp sẽ được lưu là: {JSON_OUTPUT_FILENAME}")
print(f"Số lượng quận/huyện TP.HCM được định nghĩa: {len(HCM_DISTRICTS_FULL_LIST)}")

In [ ]:
# Cell 3: Configure and Initialize undetected_chromedriver
print("Đang cấu hình Chrome Options và undetected_chromedriver ........\\n")
chrome_options_uc = Options()

# CẬP NHẬT USER-AGENT CỦA BẠN VÀO ĐÂY:
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36" 
chrome_options_uc.add_argument(f"--user-agent={USER_AGENT}")
chrome_options_uc.add_argument("--window-size=1920,1080")
# chrome_options_uc.add_argument("--headless") # Bật khi đã debug xong
# chrome_options_uc.add_argument("--disable-gpu")
# chrome_options_uc.add_argument("--no-sandbox")
# chrome_options_uc.add_argument("--disable-dev-shm-usage")
chrome_options_uc.add_argument('--disable-blink-features=AutomationControlled')

print("Chrome Options cho undetected_chromedriver đã được cấu hình.\\n")

driver = None
try:
    print("Đang khởi tạo WebDriver bằng undetected_chromedriver...")
    driver = uc.Chrome(options=chrome_options_uc, headless=False) # headless=True để chạy ẩn
    print("WebDriver đã được khởi tạo thành công bằng undetected_chromedriver!")
except Exception as e:
    print(f"Lỗi khi khởi tạo WebDriver bằng undetected_chromedriver: {e}")
    driver = None

In [ ]:
# Cell 4: Helper Function - parse_html_and_extract_data

from bs4 import BeautifulSoup # Đảm bảo import này có ở đầu file hoặc cell import
import re                     # Đảm bảo import này có
from urllib.parse import urljoin # Đảm bảo import này có

# Giả định TARGET_URL_BASE, HCM_DISTRICTS_FULL_LIST, HCM_DISTRICTS_NORMALIZED_FOR_MATCH
# đã được định nghĩa ở Cell 2
# TARGET_URL_BASE = "https://www.topcv.vn"
# HCM_DISTRICTS_FULL_LIST = [...]
# HCM_DISTRICTS_NORMALIZED_FOR_MATCH = [...]

def parse_location_details(location_label_tag, hcm_districts_full, hcm_districts_normalized):
    """
    Trích xuất city_main và district từ thẻ label chứa thông tin địa điểm.
    Ưu tiên 'data-original-title', sau đó fallback về text của 'span.city-text'.
    """
    city_main = None
    district = None
    location_raw_from_span = None # Text gốc từ span.city-text

    if not location_label_tag:
        # print("DEBUG: location_label_tag is None in parse_location_details")
        return city_main, district, location_raw_from_span

    # 1. Lấy text gốc từ span.city-text (luôn lấy để có thể dùng làm fallback hoặc ghi log)
    location_span_tag = location_label_tag.find('span', class_='city-text')
    if location_span_tag:
        location_raw_from_span = location_span_tag.get_text(strip=True)
        # print(f"DEBUG: location_raw_from_span: {location_raw_from_span}")

    # 2. Ưu tiên trích xuất từ 'data-original-title'
    tooltip_attr = location_label_tag.get('data-original-title')
    # print(f"DEBUG: tooltip_attr: {tooltip_attr}")
    if tooltip_attr:
        # data-original-title chứa HTML, cần parse nó
        tooltip_soup = BeautifulSoup(tooltip_attr, 'html.parser')
        p_tag_in_tooltip = tooltip_soup.find('p')
        if p_tag_in_tooltip:
            locations_in_tooltip = []
            current_segment = ""
            for content_part in p_tag_in_tooltip.contents:
                if isinstance(content_part, str):
                    current_segment += content_part.strip()
                elif content_part.name == 'br':
                    if current_segment:
                        locations_in_tooltip.append(current_segment)
                    current_segment = "" 
            if current_segment: 
                locations_in_tooltip.append(current_segment)
            
            # print(f"  DEBUG: Tooltip locations: {locations_in_tooltip}")

            districts_found_in_hcm = []
            is_hcm_primary = False

            for loc_detail_str in locations_in_tooltip:
                loc_detail_str_lower = loc_detail_str.lower()
                if "hồ chí minh" in loc_detail_str_lower or "hcm" in loc_detail_str_lower:
                    is_hcm_primary = True 
                    district_match = re.search(r'(?:hồ chí minh|hcm)\s*:\s*(.+)', loc_detail_str, re.IGNORECASE)
                    if district_match:
                        district_name_raw = district_match.group(1).strip()
                        district_name_normalized = district_name_raw.lower().replace("quận ", "").replace("huyện ", "").replace("thành phố ", "").replace("tp. ", "")
                        try:
                            idx = hcm_districts_normalized.index(district_name_normalized)
                            districts_found_in_hcm.append(hcm_districts_full[idx])
                        except ValueError:
                            districts_found_in_hcm.append(district_name_raw) 
                            pass 
            
            if is_hcm_primary:
                city_main = "Hồ Chí Minh"
                if districts_found_in_hcm:
                    district = ", ".join(sorted(list(set(districts_found_in_hcm))))
    
    if city_main == "Hồ Chí Minh" and not district and location_raw_from_span:
        raw_span_text_normalized = location_raw_from_span.lower().replace("quận ", "").replace("huyện ", "").replace("thành phố ", "").replace("tp. ", "")
        if raw_span_text_normalized not in ["hồ chí minh", "hcm"]: 
            try:
                idx = hcm_districts_normalized.index(raw_span_text_normalized)
                district = hcm_districts_full[idx]
            except ValueError:
                pass 

    if not city_main and location_raw_from_span:
        if "hồ chí minh" in location_raw_from_span.lower() or "hcm" in location_raw_from_span.lower():
            city_main = "Hồ Chí Minh"

    # print(f"DEBUG: Returning from parse_location_details: city={city_main}, district={district}, raw_span={location_raw_from_span}")
    return city_main, district, location_raw_from_span


def parse_html_and_extract_data(page_source_html, current_scrape_date_str):
    # print(f"  Đang phân tích HTML (kích thước: {len(page_source_html)} bytes) bằng BeautifulSoup...")
    soup = BeautifulSoup(page_source_html, 'html.parser')
    jobs_on_this_page = []
    
    job_list_container = soup.find('div', class_='box-job-list')
    if not job_list_container:
        print("  !!! Không tìm thấy container 'div.box-job-list'. Bỏ qua trang này.")
        return []

    job_item_elements = job_list_container.find_all('div', class_='job-item-search-result')

    if not job_item_elements:
        return []

    for index_job, item_element in enumerate(job_item_elements):
        job_data = {
            'job_title': None, 'job_url': None, 'company_name': None,
            'salary': None, 
            'location_raw': None, 
            'city_main': None,    
            'district': None,     
            'experience': None, 'post_date': None,
            'scrape_date': current_scrape_date_str,
        }
        
        title_block_div = item_element.find('div', class_='title-block')
        if title_block_div:
            title_h3 = title_block_div.find('h3', class_=['title', 'title highlight']) # class_ có thể là list
            if title_h3 and title_h3.find('a'):
                title_tag_a = title_h3.find('a')
                job_title_span = title_tag_a.find('span')
                job_data['job_title'] = (job_title_span.get_text(strip=True) if job_title_span 
                                         else title_tag_a.get_text(strip=True))
                job_url_relative = title_tag_a.get('href')
                if job_url_relative:
                    full_job_url = urljoin(TARGET_URL_BASE, job_url_relative)
                    # === CHUẨN HÓA URL ===
                    # Loại bỏ phần query string (tất cả những gì sau dấu '?')
                    job_data['job_url'] = full_job_url.split('?')[0]
        
        company_tag_a = item_element.find('a', class_=['company', 'company job-pro']) # class_ có thể là list
        if company_tag_a and company_tag_a.find('span', class_='company-name'):
            job_data['company_name'] = company_tag_a.find('span', class_='company-name').get_text(strip=True)
        
        salary_tag = item_element.find(['span', 'div', 'p', 'label'], class_='title-salary')
        if salary_tag:
            job_data['salary'] = salary_tag.get_text(strip=True).replace('\\n', '').replace('\\r', '').strip()
        
        # === Xử lý địa điểm (location) ===
        # KHÔI PHỤC VỀ DẠNG GỐC MÀ BẠN ĐÃ DÙNG (VÀ CÓ THỂ ĐÃ HOẠT ĐỘNG ĐÚNG)
        location_label_tag = item_element.find('label', class_='address truncate') 
        # print(f"DEBUG item {index_job}: location_label_tag find result: {location_label_tag is not None}")

        city, dist, raw_loc_span = parse_location_details(
            location_label_tag, 
            HCM_DISTRICTS_FULL_LIST, 
            HCM_DISTRICTS_NORMALIZED_FOR_MATCH 
        )
        job_data['city_main'] = city
        job_data['district'] = dist
        job_data['location_raw'] = raw_loc_span

        exp_label = item_element.find('label', class_='exp')
        if exp_label and exp_label.find('span'):
            job_data['experience'] = exp_label.find('span').get_text(strip=True)
            
        date_tag = item_element.find('label', class_='address mobile-hidden label-update')
        if date_tag:
            job_data['post_date'] = date_tag.get_text(strip=True).replace('Đăng', '').strip()
        
        if job_data['job_title'] and job_data['job_url']:
            jobs_on_this_page.append(job_data)
            
    return jobs_on_this_page

In [ ]:
# Cell 5: Helper Function - extract_max_pages_from_soup

def extract_max_pages_from_soup(soup):
    try:
        # Selector dựa trên HTML bạn cung cấp: <span id="job-listing-paginate-text"> <span class="hight-light">1 </span>/ 72 trang </span>
        pagination_text_span = soup.select_one("nav.box-pagination span#job-listing-paginate-text")
        if pagination_text_span:
            text_content = pagination_text_span.get_text(strip=True) # Ví dụ: "1 / 72 trang"
            match = re.search(r'/\s*(\d+)\s*trang', text_content)
            if match:
                return int(match.group(1))
        # print("  Không tìm thấy thông tin tổng số trang từ pagination text.") # Giảm log
    except Exception as e:
        print(f"  Lỗi khi trích xuất tổng số trang: {e}")
    return 1 # Mặc định là 1 nếu không tìm thấy hoặc lỗi

In [ ]:
# Cell 6: Main Scraping Loop with Pagination

from urllib.parse import urlparse, urlunparse, parse_qs, urlencode

all_pages_jobs_data = []
current_page_num = 1
actual_max_pages_on_site = 1

if driver:
    print(f"Bắt đầu quá trình cào dữ liệu từ: {INITIAL_TARGET_URL}")
    parsed_initial_url = urlparse(INITIAL_TARGET_URL)
    # Không cần fixed_query_params và base_path nếu mỗi lần đều parse lại INITIAL_TARGET_URL

    while True: # Vòng lặp sẽ dừng bởi các điều kiện break bên trong
        
        # --- Xây dựng URL cho trang hiện tại ---
        # Lấy các tham số query gốc từ INITIAL_TARGET_URL mỗi lần để đảm bảo tính toàn vẹn
        temp_params = parse_qs(parsed_initial_url.query)
        temp_params['page'] = [str(current_page_num)] # Đặt/Cập nhật số trang hiện tại
        
        # (Tùy chọn) Sắp xếp các tham số để URL nhất quán hơn cho việc debug
        # sorted_query_items = sorted(temp_params.items()) 
        # current_query_string = urlencode(sorted_query_items, doseq=True)
        current_query_string = urlencode(temp_params, doseq=True) # Giữ nguyên thứ tự nếu không sắp xếp

        current_url_to_scrape = urlunparse(
            (parsed_initial_url.scheme,
             parsed_initial_url.netloc,
             parsed_initial_url.path, # Sử dụng path gốc của INITIAL_TARGET_URL
             parsed_initial_url.params, 
             current_query_string,
             parsed_initial_url.fragment)
        )
        
        # --- Điều kiện dừng sớm ---
        if current_page_num > MAX_PAGES_TO_SCRAPE:
            print(f"Đã đạt giới hạn MAX_PAGES_TO_SCRAPE ({MAX_PAGES_TO_SCRAPE}). Dừng.")
            break
        if actual_max_pages_on_site > 1 and current_page_num > actual_max_pages_on_site:
            # current_page_num đã là trang tiếp theo chưa được cào, nên trang cuối cùng đã cào là current_page_num - 1
            print(f"Đã cào đến trang cuối cùng trên site ({current_page_num - 1}). Dừng.")
            break

        print(f"\n--- Đang cào dữ liệu trang {current_page_num} / {actual_max_pages_on_site if actual_max_pages_on_site > 1 else MAX_PAGES_TO_SCRAPE} ---")
        print(f"URL: {current_url_to_scrape}")
        
        page_source_html = None
        try:
            driver.get(current_url_to_scrape)
            # Tăng thời gian chờ tải trang và làm ngẫu nhiên hơn
            wait_time_seconds = random.uniform(15, 25) # Ví dụ: 15-25 giây
            print(f"  Đang đợi trang tải tối đa {wait_time_seconds:.1f} giây...")
            try:
                WebDriverWait(driver, wait_time_seconds).until(
                    EC.presence_of_element_located((By.CLASS_NAME, "box-job-list"))
                )
                print("  Container 'box-job-list' đã xuất hiện.")
            except TimeoutException:
                print(f"  Hết thời gian chờ {wait_time_seconds:.1f}s, trang có thể chưa tải đầy đủ hoặc không có 'box-job-list'.")
            
            # Mô phỏng hành vi cuộn trang
            try:
                scroll_attempts = random.randint(1, 3) # Cuộn 1-3 lần
                for i in range(scroll_attempts):
                    scroll_y = random.randint(200, 500) * (i + 1) # Cuộn xa hơn mỗi lần
                    driver.execute_script(f"window.scrollBy(0, {scroll_y});")
                    print(f"    Đã cuộn trang lần {i+1}/{scroll_attempts}.")
                    time.sleep(random.uniform(0.8, 2.2)) # Nghỉ ngắn giữa các lần cuộn
                # (Tùy chọn) Cuộn về đầu trang
                # driver.execute_script("window.scrollTo(0, 0);")
                # print("    Đã cuộn về đầu trang.")
            except Exception as e_scroll:
                print(f"  Lỗi khi cố gắng cuộn trang: {e_scroll}")

            page_source_html = driver.page_source
            print(f"  Đã lấy HTML (kích thước: {len(page_source_html)} bytes).")

        except Exception as e:
            print(f"  Lỗi nghiêm trọng khi truy cập URL {current_url_to_scrape} hoặc lấy page_source: {e}")
            # Xem xét việc có nên thoát khỏi vòng lặp hay thử lại (phức tạp hơn)
            break 

        if page_source_html:
            scrape_date_today = datetime.now().strftime('%Y-%m-%d')
            jobs_from_current_page = parse_html_and_extract_data(page_source_html, scrape_date_today) # Gọi hàm từ Cell 4
            if jobs_from_current_page:
                all_pages_jobs_data.extend(jobs_from_current_page)
                print(f"  Đã trích xuất {len(jobs_from_current_page)} jobs từ trang {current_page_num}.")
            else:
                print(f"  Không trích xuất được job nào từ trang {current_page_num}.")

            soup_for_pagination = BeautifulSoup(page_source_html, 'html.parser')
            if current_page_num == 1: 
                max_pages_from_site_temp = extract_max_pages_from_soup(soup_for_pagination) # Gọi hàm từ Cell 5
                if max_pages_from_site_temp > 1:
                     actual_max_pages_on_site = max_pages_from_site_temp
                print(f"  Phát hiện tổng số trang trên site là: {actual_max_pages_on_site}")
            
            # --- Logic dừng dựa trên nút "Next" ---
            next_page_link_tag = soup_for_pagination.select_one("ul.pagination a[rel='next'][data-href]")
            if not next_page_link_tag:
                if current_page_num < actual_max_pages_on_site and actual_max_pages_on_site > 1:
                    print(f"  Không tìm thấy link 'Next' dù chưa phải trang cuối theo site ({current_page_num}/{actual_max_pages_on_site}). Dừng.")
                else: # Bao gồm trường hợp chỉ có 1 trang hoặc đã ở trang cuối cùng thực sự
                    print("  Không tìm thấy link 'Next' page. Đây có thể là trang cuối cùng. Dừng.")
                break
        else: 
            print(f"Không lấy được page_source cho trang {current_page_num}. Dừng.")
            break
            
        current_page_num += 1
        
        # --- Nghỉ giữa các trang danh sách ---
        if current_page_num <= MAX_PAGES_TO_SCRAPE and current_page_num <= actual_max_pages_on_site : # Chỉ nghỉ nếu còn trang để cào
            # Nghỉ dài hơi sau mỗi N trang
            if (current_page_num -1) % 7 == 0 and (current_page_num -1) > 0 : # Sau mỗi 7 trang đã cào (ví dụ)
                long_sleep_duration = random.uniform(90, 240) # Nghỉ 1.5 - 4 phút
                print(f"--- ĐÃ CÀO {current_page_num -1} TRANG, NGHỈ DÀI {long_sleep_duration/60:.1f} PHÚT ---")
                time.sleep(long_sleep_duration)
            else: # Nghỉ bình thường giữa các trang
                normal_sleep_duration = random.uniform(25, 50) # Tăng lên 25-50 giây
                print(f"  Nghỉ {normal_sleep_duration:.1f} giây trước khi cào trang danh sách tiếp theo...")
                time.sleep(normal_sleep_duration)
        
    print(f"\n--- HOÀN TẤT QUÁ TRÌNH SCRAPING ---")
    
    final_pages_actually_processed = current_page_num -1
    if not all_pages_jobs_data and final_pages_actually_processed == 0 : # Nếu trang đầu tiên không có gì và dừng
         pass # final_pages_actually_processed vẫn là 0
    elif final_pages_actually_processed == 0 and all_pages_jobs_data: # Xử lý trang 1 và dừng
        final_pages_actually_processed = 1

    print(f"Tổng số trang đã thực sự xử lý: {final_pages_actually_processed}")
    print(f"Tổng số jobs đã cào được: {len(all_pages_jobs_data)}")

else:
    print("WebDriver không được khởi tạo. Bỏ qua quá trình scraping.")
    

In [ ]:
# Cell 7: Save ALL Extracted Data to JSON File and Close Webdriver
if all_pages_jobs_data:
    try:
        print(f"\nĐang chuẩn bị lưu {len(all_pages_jobs_data)} job items vào file JSON: {JSON_OUTPUT_FILENAME}")
        with open(JSON_OUTPUT_FILENAME, 'w', encoding='utf-8') as f_json:
            json.dump(all_pages_jobs_data, f_json, ensure_ascii=False, indent=4)
        print(f"Thành công! Dữ liệu đã được lưu vào file: '{JSON_OUTPUT_FILENAME}'")
        
        if len(all_pages_jobs_data) > 0:
             print("\nVí dụ 1 job đầu tiên đã trích xuất (từ tổng hợp):")
             print(json.dumps(all_pages_jobs_data[0], indent=2, ensure_ascii=False))
    except Exception as e_save:
        print(f"Lỗi khi lưu file JSON: {e_save}")
else:
     print("Không có dữ liệu job nào để lưu.")

if driver:
    try:
        # print("\nĐang đóng WebDriver...") # Giảm log
        driver.quit()
        # print("WebDriver đã được đóng thành công.") # Giảm log
        driver = None 
    except Exception as e_quit:
        print(f"Lỗi khi đóng WebDriver: {e_quit}")
# else: # Giảm log
    # print("\nWebDriver không tồn tại hoặc đã được đóng trước đó.")

In [ ]:
# Cell 8: Load Data from JSON and Initial Data Overview
df_jobs = pd.DataFrame() # Khởi tạo df rỗng
if os.path.exists(JSON_OUTPUT_FILENAME):
    try:
        df_jobs = pd.read_json(JSON_OUTPUT_FILENAME, encoding='utf-8')
        if df_jobs.empty and len(all_pages_jobs_data) > 0: # Nếu đọc file rỗng nhưng all_pages_jobs_data có dữ liệu (lỗi lưu file?)
             print("!!! Cảnh báo: Đọc file JSON trả về DataFrame rỗng, nhưng có dữ liệu trong `all_pages_jobs_data`. Kiểm tra lại quá trình lưu file.")
             df_jobs = pd.DataFrame(all_pages_jobs_data) # Thử tạo df từ biến trong memory

        if not df_jobs.empty:
            print(f"\n--- Đã đọc và tạo DataFrame với {len(df_jobs)} job items ---")
            print("\nThông tin DataFrame (df_jobs.info()):")
            df_jobs.info()
            print('\n=== 5 dòng dữ liệu đầu tiên (df_jobs.head()) ===')
            display(df_jobs.head())
            print("\n--- Kiểm tra giá trị NULL cho mỗi cột ---")
            print(df_jobs.isnull().sum())
            
            print("\n--- Một số thông tin thống kê ban đầu ---")
            print("\nPhân bổ 'city_main':")
            print(df_jobs['city_main'].value_counts(dropna=False))
            print("\nPhân bổ 'district' (cho TP.HCM nếu có):")
            if "Hồ Chí Minh" in df_jobs['city_main'].unique():
                print(df_jobs[df_jobs['city_main'] == "Hồ Chí Minh"]['district'].value_counts(dropna=False))
            else:
                print("Không có job nào tại Hồ Chí Minh để hiển thị quận.")

        else:
            print(f"File {JSON_OUTPUT_FILENAME} được đọc nhưng DataFrame vẫn rỗng. Không có dữ liệu để xử lý.")
    except Exception as e_read_json:
        print(f"Lỗi khi đọc file JSON '{JSON_OUTPUT_FILENAME}': {e_read_json}")
        print("Sẽ thử tạo DataFrame từ biến `all_pages_jobs_data` nếu có.")
        if all_pages_jobs_data:
            df_jobs = pd.DataFrame(all_pages_jobs_data)
            if not df_jobs.empty:
                 print(f"Đã tạo DataFrame từ `all_pages_jobs_data` với {len(df_jobs)} dòng.")
            else:
                 print("`all_pages_jobs_data` cũng rỗng.")
        else:
            print("Không có dữ liệu trong `all_pages_jobs_data` để tạo DataFrame.")
else:
    print(f"File {JSON_OUTPUT_FILENAME} không tồn tại. Không có dữ liệu để đọc.")

In [ ]:
# Cell 9: Detailed Processing for 'salary' column
if not df_jobs.empty:
    print("\n--- Bắt đầu xử lý cột 'salary' ---")
    df_jobs['salary_min_mil'] = np.nan
    df_jobs['salary_max_mil'] = np.nan
    df_jobs['salary_currency'] = 'triệu VNĐ' # Mặc định
    df_jobs['salary_type'] = 'Không xác định'

    def parse_salary_string_v2(salary_text):
        if pd.isna(salary_text): return np.nan, np.nan, 'triệu VNĐ', 'Không xác định'
        
        text_original = str(salary_text)
        text_lower = text_original.lower()
        
        min_sal, max_sal = np.nan, np.nan
        currency = 'triệu VNĐ'
        sal_type = 'Không xác định'

        if "thoả thuận" in text_lower or "thỏa thuận" in text_lower:
            sal_type = "Thoả thuận"
            return min_sal, max_sal, currency, sal_type

        # Phát hiện đơn vị (USD trước triệu để tránh nhầm lẫn)
        if "usd" in text_lower:
            currency = "USD"
            # Loại bỏ "usd" và các ký tự không liên quan, chỉ giữ số và dấu phân cách
            text_for_num_extract = re.sub(r'[^0-9.,\-]', '', text_lower.replace("usd",""))
        elif "triệu" in text_lower:
            currency = "triệu VNĐ"
            text_for_num_extract = re.sub(r'[^0-9.,\-]', '', text_lower.replace("triệu","").replace("vnd",""))
        else: # Mặc định là triệu nếu không có đơn vị
            text_for_num_extract = re.sub(r'[^0-9.,\-]', '', text_lower)

        # Trích xuất số, cho phép dấu phẩy là phân cách hàng nghìn và dấu chấm là thập phân
        # Hoặc ngược lại (Việt Nam thường dùng dấu chấm cho hàng nghìn, phẩy cho thập phân)
        # Ưu tiên tìm số có thể có dấu phẩy hoặc chấm làm thập phân
        numbers_found = re.findall(r'\d+[\.,]?\d*', text_for_num_extract)
        
        processed_numbers = []
        for num_str in numbers_found:
            # Chuẩn hóa: thay dấu phẩy bằng dấu chấm nếu nó là thập phân duy nhất
            # và loại bỏ dấu chấm nếu nó là hàng ngàn (ví dụ 1.000.000)
            # Đây là một bước đơn giản hóa, có thể cần phức tạp hơn cho mọi trường hợp
            num_str_cleaned = num_str.replace('.', '').replace(',', '.') if ',' in num_str else num_str.replace(',', '')
            try:
                processed_numbers.append(float(num_str_cleaned))
            except ValueError:
                # print(f"Warning: Could not convert number string '{num_str_cleaned}' from '{num_str}'")
                pass
        
        numbers = sorted(processed_numbers) # Sắp xếp để dễ lấy min/max

        if not numbers: # Không tìm thấy số nào
            return min_sal, max_sal, currency, sal_type

        # Xác định loại lương dựa trên từ khóa và số lượng
        if "tới" in text_lower or "upto" in text_lower or ("đến" in text_lower and len(numbers) == 1 and not "từ" in text_lower):
            sal_type = "Tối đa"
            if numbers: max_sal = numbers[-1] # Lấy số lớn nhất
        elif "từ" in text_lower and not ("đến" in text_lower or "tới" in text_lower or "-" in text_original): # Chỉ "Từ X"
            sal_type = "Tối thiểu"
            if numbers: min_sal = numbers[0] # Lấy số nhỏ nhất
        elif (("-" in text_original or "đến" in text_lower) and len(numbers) >= 2) or len(numbers) >=2 : # "X - Y" hoặc "Từ X đến Y"
            sal_type = "Khoảng"
            min_sal = numbers[0]
            max_sal = numbers[-1] # Số lớn nhất trong các số tìm được
            if min_sal == max_sal and len(numbers) > 1 : # ví dụ "10-10 triệu"
                 sal_type = "Cố định" # Hoặc vẫn là khoảng tùy ý
        elif len(numbers) == 1:
            sal_type = "Cố định"
            min_sal = max_sal = numbers[0]
        
        # Nếu là USD và đã có số, cần kiểm tra xem có cần nhân với tỉ giá không
        # Hiện tại chưa nhân, chỉ lưu dạng USD
        
        return min_sal, max_sal, currency, sal_type

    # Áp dụng hàm cho từng dòng
    # Sử dụng .apply() với lambda để trả về nhiều giá trị cho nhiều cột
    salary_details = df_jobs['salary'].apply(lambda x: pd.Series(parse_salary_string_v2(x), 
                                                              index=['salary_min_mil', 'salary_max_mil', 'salary_currency', 'salary_type']))
    df_jobs[['salary_min_mil', 'salary_max_mil', 'salary_currency', 'salary_type']] = salary_details

    print("\n--- DataFrame sau khi xử lý cột 'salary' (5 dòng đầu) ---")
    display(df_jobs[['salary', 'salary_min_mil', 'salary_max_mil', 'salary_currency', 'salary_type']].head())
    
    print("\nKiểu dữ liệu các cột salary mới:")
    print(df_jobs[['salary_min_mil', 'salary_max_mil']].dtypes)
    
    if df_jobs['salary_min_mil'].notna().any() or df_jobs['salary_max_mil'].notna().any():
        print("\nThống kê mô tả cho salary_min_mil và salary_max_mil (cho currency='triệu VNĐ'):")
        display(df_jobs[df_jobs['salary_currency'] == 'triệu VNĐ'][['salary_min_mil', 'salary_max_mil']].describe())
    
    print("\nPhân loại 'salary_type':")
    print(df_jobs['salary_type'].value_counts(dropna=False))
    print("\nPhân loại 'salary_currency':")
    print(df_jobs['salary_currency'].value_counts(dropna=False))
else:
    print("\nDataFrame 'df_jobs' rỗng. Bỏ qua xử lý cột 'salary'.")

In [ ]:
# Cell 10: Detailed Processing for 'post_date' column
if not df_jobs.empty:
    if 'scrape_date' not in df_jobs.columns:
        print("\nLỗi: Cột 'scrape_date' không tồn tại trong DataFrame (Cell 10).")
    else:
        print("\n--- Bắt đầu xử lý cột 'post_date' ---")
        df_jobs['calculated_post_date'] = pd.NaT # Kiểu datetime của Pandas cho ngày không xác định
        
        for index, row in df_jobs.iterrows():
            date_str_original = row['post_date']
            scrape_date_original = row['scrape_date'] 
            
            calculated_date_val = pd.NaT 

            if pd.isna(date_str_original) or pd.isna(scrape_date_original):
                df_jobs.loc[index, 'calculated_post_date'] = pd.NaT
                continue

            try:
                # Đảm bảo scrape_date_original là string trước khi parse
                scrape_datetime_object = datetime.strptime(str(scrape_date_original), '%Y-%m-%d')
            except ValueError:
                # print(f"  Lỗi định dạng scrape_date: '{scrape_date_original}' tại index {index}.")
                df_jobs.loc[index, 'calculated_post_date'] = pd.NaT
                continue

            date_str_lower = str(date_str_original).lower()

            try:
                if "hôm qua" in date_str_lower:
                    calculated_date_val = scrape_datetime_object - timedelta(days=1)
                elif "hôm nay" in date_str_lower: 
                    calculated_date_val = scrape_datetime_object
                elif "ngày trước" in date_str_lower:
                    days_match = re.search(r'(\d+)\s*ngày trước', date_str_lower)
                    if days_match:
                        calculated_date_val = scrape_datetime_object - timedelta(days=int(days_match.group(1)))
                elif "tuần trước" in date_str_lower:
                    weeks_match = re.search(r'(\d+)\s*tuần trước', date_str_lower)
                    if weeks_match:
                        calculated_date_val = scrape_datetime_object - timedelta(weeks=int(weeks_match.group(1)))
                elif "tháng trước" in date_str_lower:
                    months_match = re.search(r'(\d+)\s*tháng trước', date_str_lower)
                    if months_match:
                        # Ước lượng 1 tháng = 30.44 ngày
                        calculated_date_val = scrape_datetime_object - timedelta(days=int(int(months_match.group(1)) * 30.4375)) 
                else: # Nếu không phải dạng tương đối, thử parse trực tiếp
                    try:
                        # Thử parse với format d/m/Y hoặc m/d/Y (phổ biến)
                        parsed_dt_direct = pd.to_datetime(date_str_original, dayfirst=True, errors='coerce')
                        if pd.notna(parsed_dt_direct):
                             calculated_date_val = parsed_dt_direct
                        else: # Thử format khác nếu cần
                             parsed_dt_direct_alt = pd.to_datetime(date_str_original, errors='coerce')
                             if pd.notna(parsed_dt_direct_alt):
                                 calculated_date_val = parsed_dt_direct_alt
                    except Exception:
                        pass 
                
                if pd.notna(calculated_date_val):
                    # Chuẩn hóa về đầu ngày (00:00:00)
                    if isinstance(calculated_date_val, pd.Timestamp):
                        df_jobs.loc[index, 'calculated_post_date'] = calculated_date_val.normalize()
                    elif isinstance(calculated_date_val, datetime): # Nếu là datetime.datetime
                        df_jobs.loc[index, 'calculated_post_date'] = calculated_date_val.replace(hour=0, minute=0, second=0, microsecond=0)
            except Exception as e_date_proc:
                # print(f"  Lỗi khi xử lý post_date '{date_str_original}' tại index {index}: {e_date_proc}")
                df_jobs.loc[index, 'calculated_post_date'] = pd.NaT
        
        print("\n--- DataFrame sau khi xử lý cột 'post_date' (5 dòng đầu) ---")
        display(df_jobs[['post_date', 'scrape_date', 'calculated_post_date']].head())
        print("\nKiểu dữ liệu của cột 'calculated_post_date':")
        print(df_jobs['calculated_post_date'].dtype) 
        print(f"Số lượng ngày không xác định (NaT): {df_jobs['calculated_post_date'].isnull().sum()}")

        if df_jobs['calculated_post_date'].notnull().any():
            print("\nThống kê mô tả cho 'calculated_post_date':")
            display(df_jobs['calculated_post_date'].describe())
else:
    print("\nDataFrame 'df_jobs' rỗng. Bỏ qua xử lý cột 'post_date'.")

In [ ]:
# Cell 11: Detailed Processing for 'experience' column
if not df_jobs.empty:
    print("\n--- Bắt đầu xử lý cột 'experience' ---")
    df_jobs['experience_standardized'] = 'Không xác định' # Giá trị mặc định

    def standardize_experience_v2(exp_str):
        if pd.isna(exp_str):
            return 'Không xác định'

        exp_lower = str(exp_str).lower()

        if any(keyword in exp_lower for keyword in ["không yêu cầu", "chưa có kinh nghiệm", "no experience", "fresher", "mới tốt nghiệp"]):
            return "Không yêu cầu"
        
        # Dưới X năm / < X năm
        match_under = re.search(r'(?:dưới|<)\s*(\d+)\s*(?:năm|year)', exp_lower)
        if match_under:
            return f"< {match_under.group(1)} năm"

        # Khoảng X-Y năm / Từ X đến Y năm
        match_range = re.search(r'(?:từ\s*)?(\d+)\s*(?:-|đến)\s*(\d+)\s*(?:năm|year)', exp_lower)
        if match_range:
            return f"{match_range.group(1)}-{match_range.group(2)} năm"

        # Chính xác X năm
        match_exact = re.search(r'^(\d+)\s*(?:năm|year)', exp_lower) # ^ để khớp từ đầu
        if match_exact:
            return f"{match_exact.group(1)} năm"
        
        # Trên X năm / > X năm
        match_over = re.search(r'(?:trên|>)\s*(\d+)\s*(?:năm|year)', exp_lower)
        if match_over:
            return f"> {match_over.group(1)} năm"
            
        return exp_str # Trả về chuỗi gốc nếu không khớp mẫu nào rõ ràng

    df_jobs['experience_standardized'] = df_jobs['experience'].apply(standardize_experience_v2)

    print("\n--- DataFrame sau khi xử lý cột 'experience' (5 dòng đầu) ---")
    display(df_jobs[['experience', 'experience_standardized']].head())

    print("\nCác giá trị duy nhất trong 'experience_standardized':")
    print(df_jobs['experience_standardized'].value_counts(dropna=False))
else:
    print("\nDataFrame 'df_jobs' rỗng. Bỏ qua xử lý cột 'experience'.")
    

In [ ]:
# Cell 12: PostgreSQL Schemas, Connection, Table Creation, and Data Insertion

DB_HOST = os.getenv('DB_HOST', 'localhost') # Thêm giá trị mặc định phòng trường hợp .env thiếu
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_PORT = os.getenv('DB_PORT', '5432')

TABLE_NAME = 'multipage_topcv_jobs'

CREATE_TABLE_SQL_V2 = f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    job_id SERIAL PRIMARY KEY,
    job_title VARCHAR(500),
    job_url TEXT UNIQUE,
    company_name VARCHAR(300),
    salary_raw VARCHAR(255), 
    salary_min_mil NUMERIC(10, 2),
    salary_max_mil NUMERIC(10, 2),
    salary_currency VARCHAR(20),
    salary_type VARCHAR(50),
    location_raw TEXT,
    city_main VARCHAR(255),
    district VARCHAR(255),
    experience_raw VARCHAR(255), 
    experience_standardized VARCHAR(255),
    post_date_raw VARCHAR(50), 
    calculated_post_date DATE,
    scrape_date DATE,
    inserted_at TIMESTAMP WITHOUT TIME ZONE DEFAULT CURRENT_TIMESTAMP 
);
"""

if not df_jobs.empty:
    if not all([DB_NAME, DB_USER, DB_PASSWORD]): # Kiểm tra các biến môi trường cần thiết
        print("Lỗi: Thiếu thông tin kết nối database (DB_NAME, DB_USER, DB_PASSWORD). Kiểm tra file .env hoặc biến môi trường.")
    else:
        conn, cur = None, None
        try:
            print(f"\n--- Kết nối đến DB '{DB_NAME}' trên host '{DB_HOST}' để lưu dữ liệu ---")
            conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT)
            conn.autocommit = False 
            cur = conn.cursor()
            print("Kết nối DB thành công!")
            
            cur.execute(CREATE_TABLE_SQL_V2) # Sử dụng tên biến mới nếu có
            print(f"Bảng '{TABLE_NAME}' đã sẵn sàng.")

            df_to_insert = df_jobs.copy()
            
            db_columns_for_insert = [
                'job_title', 'job_url', 'company_name', 
                'salary_raw', 
                'salary_min_mil', 'salary_max_mil', 'salary_currency', 'salary_type',
                'location_raw', 'city_main', 'district',
                'experience_raw', 
                'experience_standardized',
                'post_date_raw', 
                'calculated_post_date', 
                'scrape_date'
            ]
            
            dataframe_columns_to_select = [
                'job_title', 'job_url', 'company_name',
                'salary', # df_jobs['salary'] -> salary_raw trong DB
                'salary_min_mil', 'salary_max_mil', 'salary_currency', 'salary_type',
                'location_raw', 'city_main', 'district',
                'experience', # df_jobs['experience'] -> experience_raw trong DB
                'experience_standardized',
                'post_date', # df_jobs['post_date'] -> post_date_raw trong DB
                'calculated_post_date',
                'scrape_date'
            ]
            
            # Đảm bảo tất cả các cột cần thiết tồn tại trong df_to_insert
            missing_cols = [col for col in dataframe_columns_to_select if col not in df_to_insert.columns]
            if missing_cols:
                print(f"!!! Lỗi: Các cột sau không tìm thấy trong DataFrame để chèn vào DB: {missing_cols}")
                raise KeyError(f"Missing columns: {missing_cols}")


            # Chuyển đổi các cột ngày tháng về đúng định dạng cho DB (nếu cần)
            for col_date in ['calculated_post_date', 'scrape_date']:
                if col_date in df_to_insert.columns:
                    df_to_insert[col_date] = pd.to_datetime(df_to_insert[col_date], errors='coerce').dt.strftime('%Y-%m-%d').replace({pd.NaT: None})

            list_of_tuples = [
                tuple(row[col_df] if pd.notna(row[col_df]) else None for col_df in dataframe_columns_to_select)
                for _, row in df_to_insert.iterrows()
            ]

            if not list_of_tuples:
                print("Không có dữ liệu để chèn vào database.")
            else:
                print(f"Đã chuẩn bị {len(list_of_tuples)} dòng dữ liệu để chèn.")
                insert_query = sql.SQL("""
                    INSERT INTO {} ({}) VALUES %s
                    ON CONFLICT (job_url) DO NOTHING; 
                """).format(
                    sql.Identifier(TABLE_NAME),
                    sql.SQL(', ').join(map(sql.Identifier, db_columns_for_insert))
                )
                
                execute_values(cur, insert_query.as_string(conn), list_of_tuples, page_size=200) # Tăng page_size
                inserted_rows = cur.rowcount
                print(f"Đã chèn/cập nhật (hoặc bỏ qua nếu trùng job_url): {inserted_rows} dòng.")
                conn.commit()
                print("Dữ liệu đã được commit vào database.")

        except psycopg2.Error as e_db:
            print(f"Lỗi PostgreSQL: {e_db}")
            if conn: conn.rollback(); print("Đã rollback các thay đổi.")
        except KeyError as e_key: # Bắt lỗi KeyError từ check missing_cols
            print(f"Lỗi KeyError khi chuẩn bị chèn vào DB: {e_key}")
        except Exception as e_general:
            print(f"Lỗi không xác định khi thao tác DB: {e_general}")
            if conn: conn.rollback(); print("Đã rollback các thay đổi.")
        finally:
            if cur: cur.close()
            if conn: conn.close(); print("Đã đóng kết nối database.")
else:
    print("\nDataFrame 'df_jobs' rỗng. Không có gì để thao tác với database (Cell 12).")

print("\n--- HOÀN TẤT TOÀN BỘ QUÁ TRÌNH ---")